# 04 Resultaten inlezen en weergeven

In dit script worden de modelresultaten weergegeven en geplot.

In [3]:
import logging
import numpy as np
import pandas as pd
import geopandas as gpd
import hkvsobekpy
from pathlib import Path
import shutil
import matplotlib.pyplot as plt
from shapely.geometry import Point

import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [4]:
%load_ext autoreload
%autoreload 2

### Inlezen meetlocaties en meetdata

Afvoermetingen

In [5]:
def inlezen_csv_met_metadata(file_path: Path):
    meta = {}
    with open(file_path, "r", encoding="utf-8") as f:
        lines = f.readlines()

    for line in lines:
        if not line.startswith("#"):
            continue
        text = line.strip("#").strip()
        if ":" in text:
            key, value = text.split(":", 1)
            meta[key.strip()] = value.strip().strip("; ")

    header_idx = next(i for i, line in enumerate(lines) if "Tijdstip (UTC);Waarde" in line)

    df = pd.read_csv(
        file_path,
        sep=";",
        skiprows=header_idx + 1,
        header=None,
        names=["Tijdstip (UTC)", "Waarde"],
        decimal=",",
        skipinitialspace=True
    )

    df["Tijdstip (UTC)"] = pd.to_datetime(df["Tijdstip (UTC)"], utc=True)
    df["time"] = df["Tijdstip (UTC)"].dt.tz_convert("Europe/Amsterdam").dt.tz_localize(None)
    df["Waarde"] = df["Waarde"].replace("---", np.nan).str.replace(",", ".").astype(float)
    df = df.set_index("time")[["Waarde"]]
    df.columns = [csv_file.stem]
    x = float(meta["Postitie X"].split(";")[0].strip("; (RD)"))
    y = float(meta["Postitie Y"].split(";")[0].strip("; (RD)"))

    return meta, Point(x,y), df

In [ ]:
dir_meetdata = Path("..\\..\\WRIJ_RR_Unpaved_methode_01_data\\20260520_meetdata_RR")
dir_afvoermetingen = Path(dir_meetdata, "afvoermetingen_korte_naam")
dir_resultaten = Path("..\\..\\WRIJ_RR_Unpaved_methode_04_resultaten\\")

In [7]:
afvoermeetlocaties = gpd.GeoDataFrame()

# afvoermetingen
csv_files = list(dir_afvoermetingen.glob("*.csv"))

afvoermetingen = pd.DataFrame()
for csv_file in csv_files:
    logging.info(f"Data van: {csv_file.stem}")
    meta, point, df = inlezen_csv_met_metadata(csv_file)
    afvoermeetlocaties = pd.concat([
        afvoermeetlocaties, 
        gpd.GeoDataFrame(
            [{"naam": csv_file.stem, "geometry": point}],
            geometry="geometry",
            crs=28992
        )
    ])
    afvoermetingen = pd.merge(afvoermetingen, df[csv_file.stem], how="outer", left_index=True, right_index=True)

afvoermeetlocaties.to_file(Path(dir_afvoermetingen, "afvoermeetlocaties.gpkg"))

In [ ]:
fig = make_subplots(rows=1, cols=1)

metingen_instroom = True
metingen = True

columns = [
    'Debietmeting_Kotten_Vosseveldseweg', 
    'Stuw Watermolen Berenschot',
    'Overlaat Berenschotbrug',
    'Overlaat Ulftseweg (Bergeslagbeek)', 
    'Stuw Pelgrim (Waalse Water)',
]

sel_afvoermetingen = afvoermetingen["2015-10":"2016-09"][columns]

for i_station, station in enumerate(sel_afvoermetingen.columns):
    fig.add_trace(
        go.Scatter(
            x=sel_afvoermetingen.index, 
            y=sel_afvoermetingen[station], 
            mode="markers", 
            name=station,
        ),
        row=1, col=1
    )

fig.update_layout(
    template="simple_white",
    title="Gemeten afvoeren",
    margin=dict(l=20, r=20, t=40, b=20),
    height=600,
)
fig.update_xaxes(
    showgrid=True, 
    gridcolor="lightgray", 
)
fig.update_yaxes(
    showgrid=True, 
    gridcolor="lightgray", 
    range=[0, 14]
)
fig.write_html(Path(dir_resultaten, f"oude_ijssel_afvoermetingen.html"), include_plotlyjs="cdn")
fig.show()

In [9]:
afvoermeetlocaties_gebieden = {
    1: {"in": [], "uit": ["Stuw Pelgrim (Waalse Water)"], "gmw_bro_id": []},
    2: {"in": [], "uit": ["Overlaat Ulftseweg (Bergeslagbeek)"], "gmw_bro_id": []},
    3: {"in": ["Debietmeting_Kotten_Vosseveldseweg"], "uit": ["Stuw Watermolen Berenschot"], "gmw_bro_id": []}
    # 3: {"in": ["Debietmeting_Kotten_Vosseveldseweg"], "uit": ["Overlaat Berenschotbrug"], "gmw_bro_id": []}
}

### Selecteer welke modelresultaten (gebieden, scenario’s en periode) worden geanalyseerd

In [ ]:
# path to the package containing the data
dir_model_basis = Path("..\\..\\WRIJ_RR_Unpaved_methode_03_modellen\\")

# gebied = 0 # Oude IJssel
# gebied = 1 # West
# gebied = 2 # Centraal
# gebied = 3 # Oost

gebieden = [1, 2, 3]
scenarios = ["REF", "SCEN"]

runs = {
    "OY_0_kD5_L4": {"name": "kD(5d) L=4xl", "color": "#0072B2"},                                        # blauw
    "OY_1_kD20_L4": {"name": "kD(20d) L=4xl", "color": "#D55E00"},                                      # oranje
    "OY_2_kD20_L2": {"name": "kD(20d) L=2xl", "color": "#009E73"},                                      # groen
    "OY_3_kD20_L4_W": {"name": "kD(20d) L=4xl Wh", "color": "#56B4E9"},                                 # lichtblauw
    "OY_4_kD20_L4_W_InfMax": {"name": "kD(20d) L=2xl Wh InfMax", "color": "#F0E442"},                   # geel
    "OY_5_kD20_L4_W_InfMax_InitGWS": {"name": "kD(20d) L=2xl Wh InfMax InitGWS", "color": "#E69F00"},   # goud/oranje
}

# LONG TEST
start_date = "2014-07-1"
end_date = "2016-09-30"
seizoenen = ["zomer", "winter", "winter", "zomer"]
date_range = pd.date_range(start_date, end_date, freq="3MS")

In [15]:
simulations_total = pd.DataFrame()

for run_name in runs.keys():
    for gebied in gebieden:
        for scenario in scenarios:

            simulaties = pd.DataFrame()
            simulaties["start_date"] = date_range
            simulaties["end_date"] = simulaties["start_date"].shift(-1)
            simulaties.loc[simulaties.index[-1],"end_date"] = pd.to_datetime(end_date)
            simulaties["seizoen"] = (seizoenen * 100)[:len(date_range)]
            simulaties["scenario"] = scenario
            simulaties["gebied"] = gebied
            simulaties["run_name"] = run_name
            simulaties["restart_in"] = [0] + [1] * len(simulaties.index[1:])

            simulaties["model_name"] = simulaties.apply(lambda x: f"{run_name[:4]}_gebied{gebied}_{scenario}_rr_{pd.to_datetime(x.start_date).strftime('%Y%m%d')}_{pd.to_datetime(x.end_date).strftime('%Y%m%d')}", axis=1)
            simulations_total = pd.concat([simulations_total, simulaties])

In [19]:
simulations_total.iloc[20:40]

,start_date,end_date,seizoen,scenario,gebied,run_name,restart_in,model_name
0,2015-07-01,2015-10-01,zomer,REF,3,OY_0_kD5_L4,0,OY_0_gebied3_REF_rr_20150701_20151001
1,2015-10-01,2016-01-01,winter,REF,3,OY_0_kD5_L4,1,OY_0_gebied3_REF_rr_20151001_20160101
2,2016-01-01,2016-04-01,winter,REF,3,OY_0_kD5_L4,1,OY_0_gebied3_REF_rr_20160101_20160401
3,2016-04-01,2016-07-01,zomer,REF,3,OY_0_kD5_L4,1,OY_0_gebied3_REF_rr_20160401_20160701
4,2016-07-01,2016-09-30,zomer,REF,3,OY_0_kD5_L4,1,OY_0_gebied3_REF_rr_20160701_20160930
0,2015-07-01,2015-10-01,zomer,SCEN,3,OY_0_kD5_L4,0,OY_0_gebied3_SCEN_rr_20150701_20151001
1,2015-10-01,2016-01-01,winter,SCEN,3,OY_0_kD5_L4,1,OY_0_gebied3_SCEN_rr_20151001_20160101
2,2016-01-01,2016-04-01,winter,SCEN,3,OY_0_kD5_L4,1,OY_0_gebied3_SCEN_rr_20160101_20160401
3,2016-04-01,2016-07-01,zomer,SCEN,3,OY_0_kD5_L4,1,OY_0_gebied3_SCEN_rr_20160401_20160701
4,2016-07-01,2016-09-30,zomer,SCEN,3,OY_0_kD5_L4,1,OY_0_gebied3_SCEN_rr_20160701_20160930


INLEZEN ALLE RUNS, GEBIEDEN, SCENARIOS

In [20]:
total_link_flows = pd.DataFrame()

for run, run_dict in runs.items():
    print(run)
    for gebied in gebieden:
        for scenario in scenarios:
            simulaties = simulations_total[(simulations_total["run_name"]==run) & (simulations_total["gebied"]==gebied) & (simulations_total["scenario"]==scenario)]
            link_flows = pd.DataFrame()
            for index, simulatie in simulaties.iterrows():
                print(run  + " - " + str(simulatie.gebied) + " - " + simulatie.scenario + " - " + simulatie.model_name)

                dir_model = Path(dir_model_basis, run, f"gebied_{simulatie.gebied}", simulatie.scenario)
                unpaved_rr_file = "3blinks.his"

                path_unpaved_rr_file = Path(dir_model, simulatie.model_name, "rr", unpaved_rr_file)
                if not path_unpaved_rr_file.exists():
                    print(f"File {path_unpaved_rr_file} does not exist. Skipping.")
                    continue
                rr_his = hkvsobekpy.read_his.ReadMetadata(path_unpaved_rr_file)
                rr_results_link_flow = rr_his.DataFrame()['Link flow [m3/s]    ']
                link_flows = pd.concat([link_flows, rr_results_link_flow])
            
            if not link_flows.empty:
                total_link_flows[f"{run}_{gebied}_{scenario}"] = link_flows.sum(axis=1)

OY_0_kD5_L4
OY_0_kD5_L4 - 1 - REF - OY_0_gebied1_REF_rr_20150701_20151001
OY_0_kD5_L4 - 1 - REF - OY_0_gebied1_REF_rr_20151001_20160101
OY_0_kD5_L4 - 1 - REF - OY_0_gebied1_REF_rr_20160101_20160401
OY_0_kD5_L4 - 1 - REF - OY_0_gebied1_REF_rr_20160401_20160701
OY_0_kD5_L4 - 1 - REF - OY_0_gebied1_REF_rr_20160701_20160930
File ..\..\WRIJ_RR_Unpaved_methode_03_modellen\OY_0_kD5_L4\gebied_1\REF\OY_0_gebied1_REF_rr_20160701_20160930\rr\3blinks.his does not exist. Skipping.
OY_0_kD5_L4 - 1 - SCEN - OY_0_gebied1_SCEN_rr_20150701_20151001
OY_0_kD5_L4 - 1 - SCEN - OY_0_gebied1_SCEN_rr_20151001_20160101
OY_0_kD5_L4 - 1 - SCEN - OY_0_gebied1_SCEN_rr_20160101_20160401
OY_0_kD5_L4 - 1 - SCEN - OY_0_gebied1_SCEN_rr_20160401_20160701
OY_0_kD5_L4 - 1 - SCEN - OY_0_gebied1_SCEN_rr_20160701_20160930
File ..\..\WRIJ_RR_Unpaved_methode_03_modellen\OY_0_kD5_L4\gebied_1\SCEN\OY_0_gebied1_SCEN_rr_20160701_20160930\rr\3blinks.his does not exist. Skipping.
OY_0_kD5_L4 - 2 - REF - OY_0_gebied2_REF_rr_20150701_2

### Plot het resultaat

TODO
- gemeten afvoeren gebied 3
- initiële waterstanden
- L=2x kleine l ipv L = 4x kleine l
- grafieken grondwaterstanden van alle meetpunten plus kaartje
- dynamische grafieken

- grafieken referentie vs scenario: aangepaste kD met initiële grondwaterstanden
- harm maakt overzicht van uren, inspanningen en overdracht.

wibo gaat scenario aanpassen en dan de hele trein weer doen.

In [ ]:
start_plot = "2015-07-01"
end_plot = "2016-09-30"

selectie_gebieden = {
    1: {"ymax": 10}, 
    2: {"ymax": 2},
    3: {"ymax": 10},
}

selectie_runs = runs

selectie_scenario = {
    "REF": {"linestyle": "solid", "visible": True}, 
    "SCEN": {"linestyle": "dash", "visible": False},
}
include_metingen = True

fig = make_subplots(
    rows=len(selectie_gebieden), cols=1,
    shared_xaxes=True,
    vertical_spacing=0.03
)

metingen_instroom = True
metingen = True

for i_gebied, gebied in enumerate(selectie_gebieden.keys()):

    if include_metingen:
        
        instroompunten = afvoermeetlocaties_gebieden[gebied]["in"]
        instroom = afvoermetingen[start_date:end_date][instroompunten].sum(axis=1)
        instroom[instroom<=0.0] = np.nan

        uitstroompunten = afvoermeetlocaties_gebieden[gebied]["uit"]
        uitstroom = afvoermetingen[start_date:end_date][uitstroompunten].sum(axis=1)
        uitstroom[uitstroom<=0.0] = np.nan

        if instroompunten:
            verschil = uitstroom-instroom.shift(4)
            verschil[verschil<=0.0] = np.nan

            fig.add_trace(
                go.Scatter(
                    x=instroom.index, 
                    y=instroom, 
                    mode="markers", 
                    name="Gemeten instroom gebied 3",
                    showlegend=metingen_instroom,
                    legendgroup="uitstroom-instroom",
                    visible="legendonly",
                    marker=dict(color="purple")
                ),
                row=i_gebied+1, col=1
            )
            fig.add_trace(
                go.Scatter(
                    x=uitstroom.index, 
                    y=uitstroom, 
                    mode="markers", 
                    name="Gemeten uitstroom gebied 3",
                    showlegend=metingen_instroom,
                    legendgroup="uitstroom-instroom",
                    visible="legendonly",
                    marker=dict(color="lightgreen")
                ),
                row=i_gebied+1, col=1
            )
            fig.add_trace(
                go.Scatter(
                    x=uitstroom.index, 
                    y=verschil,
                    mode="markers", 
                    name="Afgeleide afvoer gebied 3 (4 uur shift)",
                    showlegend=metingen,
                    visible="legendonly",
                    legendgroup="uitstroom-instroom",
                    marker=dict(color="black")
                ),
                row=i_gebied+1, col=1
            )
            metingen_instroom = False

        if gebied == 3:
            uitstroom = verschil/1.45
            name = f"Afvoer uit gebied {gebied} (afgeleid+gecorrigeerd)"
        else:
            name = f"Afvoer uit gebied {gebied} (metingen)"


        fig.add_trace(
            go.Scatter(
                x=uitstroom.index, 
                y=uitstroom,
                mode="markers", 
                name=name,
                showlegend=metingen,
                legendgroup="metingen",
                marker=dict(color="black")
            ),
            row=i_gebied+1, col=1
        )

for i_gebied, gebied in enumerate(selectie_gebieden.keys()):

    total_link_flows_figure = total_link_flows[start_date:end_date]
    for scenario, scenario_dict in selectie_scenario.items():
        for run, run_dict in selectie_runs.items():
            data = total_link_flows_figure[f"{run}_{gebied}_{scenario}"].copy()
            fig.add_trace(
                go.Scatter(
                    x=data.index, 
                    y=data, 
                    mode="lines", 
                    name=f"{scenario} - {run_dict['name']}",
                    legendgroup=f"{run}_{scenario}",
                    showlegend=True if i_gebied==0 else False,
                    visible=True if (scenario_dict["visible"] and "_restart" in run) else "legendonly",
                    line=dict(dash=scenario_dict["linestyle"], color=run_dict["color"])
                ),
                row=i_gebied+1, col=1
            )

fig.update_layout(
    template="simple_white",
    title="RR-modellering Oude IJssel - Afvoer voor pilotgebieden 1/2/3",
    margin=dict(l=20, r=20, t=40, b=20),
    height=350*len(selectie_gebieden),
    # legend=dict(
    #     groupclick="togglegroup",
    #     orientation="h",
    #     yanchor="top",
    #     y=-0.05,
    #     xanchor="center",
    #     x=0.5,
    #     entrywidth=250,
    #     entrywidthmode="pixels"
    # ),
)

fig.update_xaxes(
    showgrid=True, 
    gridcolor="lightgray", 
    range=[start_plot, end_plot]
)
for i_gebied, (gebied, gebied_dict) in enumerate(selectie_gebieden.items()):
    fig.update_yaxes(
        row=i_gebied+1, 
        col=1, 
        range=[0, gebied_dict["ymax"]],
        showgrid=True,
        gridcolor="lightgray",
        title_text=f"Afvoer gebied {gebied} [m3/s]"
    )

fig.write_html(Path(dir_resultaten, f"oude_ijssel_pilot_gebieden_afvoeren.html"), include_plotlyjs="cdn")
fig.show()